In [23]:
import sqlglot
from sqlglot import expressions as exp
from sqlglot.errors import ParseError


def validate_sql(sql: str):
    errors = []
    risk_flags = []

    try:
        statements = sqlglot.parse(sql, dialect="postgres")
    except ParseError as exc:
        return {
            "is_valid": False,
            "errors": [f"SQL syntax error: {exc}"],
            "risk_flags": ["syntax_error"],
        }

    if len(statements) != 1:
        errors.append("Only one SQL statement is allowed.")
        risk_flags.append("multiple_statements")

    tree = statements[0]

    if not isinstance(tree, exp.Select):
        errors.append(
            f"Only SELECT queries are allowed. Parsed statement type: {type(tree).__name__}."
        )
        risk_flags.append("not_select_query")

    return {
        "is_valid": len(errors) == 0,
        "errors": errors,
        "risk_flags": risk_flags,
        "parsed_type": type(tree).__name__,
    }

# - only SELECT queries allowed
# - block DROP, DELETE, UPDATE, INSERT, ALTER, TRUNCATE, CREATE
# - block multiple SQL statements
# - require LIMIT for raw row outputs
# - allow aggregation queries without LIMIT

In [24]:
sql = """
    WITH AvgSalaryCTE (averageValue) AS (
        SELECT AVG(Salary)
        FROM Employees
    )
    SELECT 
        EmployeeID,
        Name, 
        Salary 
    FROM 
        Employees 
    WHERE 
        Salary > (SELECT averageValue FROM AvgSalaryCTE);
"""
validate_sql(sql)

{'is_valid': True, 'errors': [], 'risk_flags': [], 'parsed_type': 'Select'}

In [25]:
validate_sql(sql = "oof;")

{'is_valid': False,
 'errors': ['Only SELECT queries are allowed. Parsed statement type: Column.'],
 'risk_flags': ['not_select_query'],
 'parsed_type': 'Column'}